# Jour 1 · Évaluer et fiabiliser un RAG


## « La réponse a l'air bonne » n'est pas une mesure

Une démonstration réussie ne prouve pas que le système fonctionne sur d'autres formulations. Nous préparons les questions, la source attendue et le comportement attendu avant d'observer les résultats.


### Deux questions à poser, toujours dans cet ordre

1. **Recherche** : le bon passage est-il remonté ?
2. **Génération** : la réponse respecte-t-elle ce passage ?

Si le bon passage manque, modifier le prompt ne réparera pas la recherche. S'il est présent mais que la réponse le déforme, nous examinons alors le prompt ou le modèle. Cette séparation évite de corriger le mauvais composant.


![Trois niveaux d'évaluation d'un RAG](../../assets/jour_01/04_trois_niveaux_evaluation.png)

*La recherche, la génération et l'expérience utilisateur ont des causes d'échec différentes.*


In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Racine du cours :", ROOT)


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


## Recharger la base persistée

Si le notebook d'ingestion n'a pas encore créé la base, la cellule suivante la reconstruit depuis les fichiers. Le notebook reste ainsi autonome.


In [ ]:
import hashlib

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

database_dir = OUTPUTS_DIR / "chroma_helpdesk"
vectorstore = Chroma(
    collection_name="helpdesk_documents",
    embedding_function=embeddings,
    persist_directory=str(database_dir),
)

if len(vectorstore.get()["ids"]) == 0:
    loaded = []
    for path in sorted(DOCUMENTS_DIR.glob("*.md")):
        if path.name == "README.md":
            continue
        docs = TextLoader(str(path), encoding="utf-8").load()
        for doc in docs:
            doc.metadata["source"] = path.name
        loaded.extend(docs)

    pdf_path = DOCUMENTS_DIR / "guide_escalade_p1.pdf"
    pdf_docs = PyPDFLoader(str(pdf_path)).load()
    for doc in pdf_docs:
        doc.metadata["source"] = pdf_path.name
    loaded.extend(pdf_docs)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=450,
        chunk_overlap=80,
        separators=["\n## ", "\n\n", "\n", " ", ""],
    )
    chunks = splitter.split_documents(loaded)
    ids = []
    for chunk in chunks:
        raw = f"{chunk.metadata.get('source')}|{chunk.metadata.get('page', '')}|{chunk.page_content}"
        ids.append(hashlib.sha256(raw.encode("utf-8")).hexdigest())
    vectorstore.add_documents(chunks, ids=ids)

print("Chunks disponibles :", len(vectorstore.get()["ids"]))


## Étape 1 — définir les cas de test

Un petit jeu utile contient plusieurs comportements : question directe, reformulation, question de sécurité et question hors périmètre.


![Catégories de questions pour évaluer un RAG](../../assets/jour_01/04_matrice_tests.png)

*Les résultats attendus sont définis avant l'exécution pour éviter de juger après coup.*


In [ ]:
cas_tests = [
    {
        "question": "Quel portail GlobalProtect faut-il saisir ?",
        "source_attendue": "procedure_vpn.md",
        "type": "directe",
    },
    {
        "question": "Mon écran externe ne montre plus rien.",
        "source_attendue": "procedure_ecran.md",
        "type": "reformulation",
    },
    {
        "question": "Que faire avant de retirer une feuille bloquée ?",
        "source_attendue": "procedure_imprimante.md",
        "type": "directe",
    },
    {
        "question": "Puis-je communiquer mon code MFA au support ?",
        "source_attendue": "politique_securite.md",
        "type": "sécurité",
    },
]
print("Cas préparés :", len(cas_tests))


### À vous de jouer — ajouter un cas

Ajoutez une reformulation avec sa source attendue. La cellule reste exécutable avant votre modification.


In [ ]:
# Correction — ajouter une reformulation sur l'écran
nouveau_cas = {
    "question": "Mon moniteur est alimenté mais ne montre aucune image.",
    "source_attendue": "procedure_ecran.md",
    "type": "reformulation",
}
cas_tests.append(nouveau_cas)
print("Nombre de cas :", len(cas_tests))


## Étape 2 — mesurer le retrieval

Nous vérifions si la source attendue apparaît dans les `k` résultats.

```text
hit = 1 si la source est retrouvée, sinon 0
hit@k moyen = moyenne des hits
```

Cet indicateur ne mesure pas encore la qualité de la réponse rédigée.


### Exemple concret de `hit@k`

Avec `k=2`, chaque question autorise deux passages candidats. Si la source attendue apparaît parmi ces deux passages pour 4 questions sur 5, alors `hit@2 = 4 / 5 = 80 %`.

Cette mesure répond seulement à « avons-nous retrouvé la bonne source ? ». Elle ne dit pas encore si le LLM a rédigé une réponse correcte.


In [ ]:
def evaluer_retrieval(store, cas, k=2):
    lignes = []
    for item in cas:
        docs = store.similarity_search(item["question"], k=k)
        sources = [doc.metadata.get("source") for doc in docs]
        lignes.append({
            "question": item["question"],
            "attendue": item["source_attendue"],
            "retrouvees": sources,
            "hit": int(item["source_attendue"] in sources),
        })
    return lignes


resultats = evaluer_retrieval(vectorstore, cas_tests, k=2)
for ligne in resultats:
    print("\n", ligne["question"])
    print(" attendue  :", ligne["attendue"])
    print(" retrouvées:", ligne["retrouvees"])
    print(" hit       :", ligne["hit"])

hit_at_2 = sum(ligne["hit"] for ligne in resultats) / len(resultats)
print(f"\nhit@2 = {hit_at_2:.0%}")


## Étape 3 — générer avec traçabilité

Une source fiable est attachée depuis les métadonnées des chunks réellement utilisés. Elle ne doit pas être générée librement par le LLM.


![Lien entre réponse, métadonnée et chunk original](../../assets/jour_01/04_tracabilite_sources.png)

*L'utilisateur peut remonter de la réponse au passage source réellement sélectionné.*


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


def format_docs(docs):
    return "\n\n".join(
        f"[Source : {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Réponds uniquement avec le contexte. Si l'information manque, réponds : "
        "Je ne trouve pas cette information dans la documentation fournie.\n\n"
        "CONTEXTE :\n{contexte}",
    ),
    ("human", "{question}"),
])
generation = prompt | llm | StrOutputParser()


def repondre_avec_sources(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    reponse = generation.invoke({
        "contexte": format_docs(docs),
        "question": question,
    })
    sources = sorted({doc.metadata.get("source", "source inconnue") for doc in docs})
    return {"reponse": reponse, "sources": sources, "documents": docs}


resultat = repondre_avec_sources("Quel portail dois-je saisir dans GlobalProtect ?")
print(resultat["reponse"])
print("Sources :", resultat["sources"])


## Étape 4 — tester le refus

Un retriever renvoie toujours les meilleurs candidats disponibles. Pour une question hors périmètre, nous contrôlons les passages remontés et la capacité du modèle à ne pas utiliser ses connaissances générales.


In [ ]:
test_refus = repondre_avec_sources("Qui a gagné la dernière Coupe du monde de football ?")
print("RÉPONSE :", test_refus["reponse"])
print("SOURCES CANDIDATES :", test_refus["sources"])


## Limites de cette première évaluation

- `hit@k` vérifie la source, pas nécessairement le meilleur paragraphe.
- Une présence de mots-clés ne suffit pas à prouver la fidélité.
- Un seuil de distance doit être calibré sur notre modèle et nos données.
- Une évaluation sérieuse conserve questions, attentes, versions et résultats.

Le réflexe essentiel est de **définir les comportements avant de regarder les réponses**.


## À retenir

- Évaluer d'abord le retrieval, puis la génération.
- Conserver plusieurs types de questions.
- Attacher les sources depuis les métadonnées.
- Examiner les documents lors d'un échec.
- Ne jamais transformer un seuil observé en vérité universelle.
